# Titanic Survival Prediction (Binary Classification)

This session predicts whether a passenger survived the Titanic disaster or not.
This is called **binary classification**, because there are only two possible answers:
**Survived** or **Did not survive**.

We will use a real dataset of Titanic passengers to find patterns (for example, did age or
class matter?) and train a model to predict survival.

Instructions for running this notebook:
- Run each cell in order using Shift + Enter.
- Cells marked LIVE CODING are meant to be written together with the audience during the session.

## 1. Load the Dataset

**About this dataset:** this is a real dataset of Titanic passengers. Each row is one passenger.
The main columns we will use are `age`, `sex`, `pclass` (ticket class), and `fare` (ticket price),
to predict `survived`. Here is what the columns mean:

- `survived`: 0 = did not survive, 1 = survived (this is what we want to predict)
- `pclass`: ticket class, 1 = first class (best), 2 = second class, 3 = third class
- `sex`: male or female
- `age`: age of the passenger in years
- `fare`: how much the passenger paid for their ticket, in pounds
- `sibsp`: number of siblings/spouses aboard
- `parch`: number of parents/children aboard

In [ ]:
import seaborn as sns

# Load a real dataset of Titanic passengers
titanic = sns.load_dataset("titanic")
titanic.head()


In [ ]:
# Basic info about the dataset
print("Rows and columns:", titanic.shape)
print()
print(titanic.info())


For this session, we will only keep the columns we actually need, to keep things simple.

In [ ]:
# Keep only the columns we need
df = titanic[["survived", "pclass", "sex", "age", "fare", "sibsp", "parch"]].copy()
print("Shape:", df.shape)
df.head()


## 2. Simple Data Cleaning

Before using data, always check for missing values.

In [ ]:
# Check for missing values in each column
df.isnull().sum()


`age` has some missing values. A simple and common fix is to fill missing ages with the
median (typical) age, instead of throwing those rows away.

In [ ]:
# Fill missing age values with the median age
df["age"] = df["age"].fillna(df["age"].median())

# Confirm there are no missing values left
df.isnull().sum()


`sex` is text ("male" or "female"). A model only understands numbers, so we convert it:
female = 1, male = 0.

In [ ]:
# Convert sex into a number: female = 1, male = 0
df["sex_score"] = df["sex"].map({"female": 1, "male": 0})
df.head()


## 3. Simple Visualization

Let's look at how survival relates to passenger class.

In [ ]:
import matplotlib.pyplot as plt

survival_by_class = df.groupby("pclass")["survived"].mean()

plt.bar(survival_by_class.index, survival_by_class.values)
plt.title("Survival Rate by Ticket Class")
plt.xlabel("Ticket Class (1 = best)")
plt.ylabel("Fraction that Survived")
plt.show()


### LIVE CODING 1

Draw a bar chart showing the survival rate grouped by `sex` instead of `pclass`.
Hint: use `df.groupby("sex")["survived"].mean()`.

In [ ]:
import matplotlib.pyplot as plt

survival_by_sex = df.groupby(...)[...].mean()

plt.bar(survival_by_sex.index, survival_by_sex.values)
plt.title("Survival Rate by Sex")
plt.xlabel("Sex")
plt.ylabel("Fraction that Survived")
plt.show()


## 4. Building a Classification Model

We will train a simple **Logistic Regression** model to predict `survived` from `pclass`
and `sex_score` only. One important idea: even though this is called "regression", Logistic
Regression is actually used for classification. It predicts a probability between 0 and 1
(for example, 0.8 = 80% chance of survival), and we round that to 0 or 1.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# Input: pclass and sex_score      Output: survived
X = df[["pclass", "sex_score"]]
y = df["survived"]

# Split into training data and testing data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create and train the model
model = LogisticRegression()
model.fit(X_train, y_train)

print("Model trained.")


In [ ]:
# Check how well the model performs on data it has not seen before
from sklearn.metrics import accuracy_score, confusion_matrix

predictions = model.predict(X_test)

acc = accuracy_score(y_test, predictions)
cm = confusion_matrix(y_test, predictions)

print("Accuracy:", round(acc, 3))
print()
print("Confusion Matrix:")
print(cm)


**What these numbers mean, in simple terms:**

- **Accuracy** - out of all predictions, what fraction did the model get right?
  0.80 means the model was correct 80% of the time.
- **Confusion Matrix** - a small table comparing what the model predicted against what
  actually happened. It looks like this:

  |                    | Predicted: Did not survive | Predicted: Survived |
  |--------------------|-----------------------------|----------------------|
  | Actual: Did not survive | Correct (top-left)     | Mistake                |
  | Actual: Survived        | Mistake                 | Correct (bottom-right) |

  The top-left and bottom-right numbers are correct predictions. The other two are mistakes,
  in two different directions. This matters because in real problems, one kind of mistake
  can be worse than the other (for example, in medicine, missing a real illness is worse
  than a false alarm).

In [ ]:
# Predict the survival of a new passenger
# pclass = 1 (first class), sex_score = 1 (female)
new_passenger = [[1, 1]]
predicted = model.predict(new_passenger)
predicted_probability = model.predict_proba(new_passenger)

print("Prediction (0 = did not survive, 1 = survived):", predicted[0])
print("Probability of survival:", round(predicted_probability[0][1], 3))


### LIVE CODING 2

Predict survival for a passenger of your choice. Try `pclass = 3` (third class) and
`sex_score = 0` (male), and compare the result to the example above.

In [ ]:
# Step 1: choose pclass and sex_score
new_passenger = [[..., ...]]

# Step 2: predict
predicted = model.predict(...)
predicted_probability = model.predict_proba(...)

print("Prediction:", predicted[0])
print("Probability of survival:", predicted_probability[0][1])


## 5. Improving the Model

Our model only looks at `pclass` and `sex_score`. But age and fare might matter too.
Let's add `age`, `fare`, `sibsp`, and `parch` to the model and see if predictions improve.

In [ ]:
# Train a new model using more columns
X_improved = df[["pclass", "sex_score", "age", "fare", "sibsp", "parch"]]
y_improved = df["survived"]

X_train2, X_test2, y_train2, y_test2 = train_test_split(X_improved, y_improved, test_size=0.2, random_state=42)

improved_model = LogisticRegression(max_iter=1000)
improved_model.fit(X_train2, y_train2)

print("Improved model trained.")


In [ ]:
# Compare the improved model to our first model
improved_predictions = improved_model.predict(X_test2)
acc_improved = accuracy_score(y_test2, improved_predictions)

print("Original model (pclass + sex)              Accuracy:", round(acc, 3))
print("Improved model (+ age, fare, sibsp, parch)  Accuracy:", round(acc_improved, 3))


**What happened:** adding more relevant information about each passenger gave the model
more to work with. This is the same idea from before: a model is often only as good as
the information it is given. Note that adding more columns does not always help; sometimes
it adds noise instead of useful signal. Checking accuracy before and after, like we just did,
is how you find out.

## Summary

Covered in this session:
- Loading a real dataset
- Checking for and handling missing values
- Converting text categories into numbers
- Visualizing survival rate across groups
- Training a simple Logistic Regression classifier
- Measuring performance with accuracy and a confusion matrix
- Using the model to predict a new passenger's outcome
- Improving the model by adding more relevant information

Reference: https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression